In [3]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
carregador_pasta = DirectoryLoader(
    "./data/documentos_fonte", glob="*.pdf", loader_cls=PyMuPDFLoader
)
documentos = carregador_pasta.load()
  

print(f" {len(documentos)}")

 11


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
all_splits = text_splitter.split_documents(documentos)
print(f"Split documentation into {len(all_splits)} chunks.")

Split documentation into 55 chunks.


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\User\Desktop\voa-bank-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3961.50it/s]


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore


vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=all_splits)
print(f"Indexed {len(all_splits)} chunks.")

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vector_store.as_retriever(search_kwargs={"k": 6})

llm = ChatOllama(model="llama3.1", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Você é um assistente do Voa Bank. Responda a pergunta do colaborador
usando APENAS o contexto abaixo.

Leia todos os trechos do contexto com atenção antes de responder.
Se o contexto afirmar ou negar algo relacionado à pergunta, isso é uma resposta válida
e você deve respondê-la normalmente — inclusive se a resposta for "não" ou uma negação.
Só diga que não encontrou a informação se o assunto da pergunta realmente não for
mencionado em nenhum trecho do contexto.

Quando o contexto descrever uma REGRA GERAL e também uma EXCEÇÃO a essa regra,
responda com base na regra geral primeiro, e mencione a exceção em seguida.
Não responda "sim" apenas porque uma exceção existe, se a regra geral for "não".

Sempre justifique sua resposta com uma frase curta baseada no contexto, para que o
colaborador possa verificar a fonte da informação. Não responda apenas "sim" ou "não"
sem explicação.

Base de informações:
{base}

Pergunta: {pergunta}

Resposta:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"base": retriever | format_docs, "pergunta": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
pergunta = input("Escreva sua pergunta: ")
resposta = rag_chain.invoke(pergunta)
print(resposta)